In [ ]:
# =========================
# nb_exception_triage
# =========================
# Phase 9, SC-15: Triage validation exceptions and assign root causes + actions
# Classifies failures and recommends remediation strategies

# ---------- Parameters ----------
period = "2026-03"
run_id = "manual_test_run"

# ---------- Imports ----------
from pyspark.sql import functions as F
import sys
import json
import hashlib

# ---------- Pre-flight checks ----------
try:
    mssparkutils
except NameError:
    print("[ERROR] mssparkutils not available - not running in Fabric context")
    sys.exit(1)

try:
    if not period or not period.strip():
        raise ValueError("period parameter is empty")
    if not run_id or not run_id.strip():
        raise ValueError("run_id parameter is empty")
    print(f"[nb_exception_triage] START period={period}, run_id={run_id}")
except ValueError as e:
    print(f"[ERROR] Invalid parameters: {e}")
    sys.exit(1)

if not spark.catalog.tableExists("validation_events") or not spark.catalog.tableExists("ai_triage_labels"):
    print("[ERROR] Required tables do not exist")
    sys.exit(1)

# Load shared_ai_utils
sys.path.append('/lakehouse/default/Files/config')
try:
    from shared_ai_utils import load_ai_config
    ai_enabled = True
except ImportError as e:
    print(f"[WARNING] shared_ai_utils not found: {e}")
    print("[nb_exception_triage] AI not available; skipping")
    mssparkutils.notebook.exit("AI_DISABLED")

# Load AI config
try:
    ai_client = load_ai_config('/lakehouse/default/Files/config/azure_openai_config.json')
    if ai_client is None:
        raise RuntimeError("AI config returned None")
except Exception as e:
    print(f"[ERROR] Failed to load AI config: {type(e).__name__}: {e}")
    mssparkutils.notebook.exit("CONFIG_ERROR")

# ---------- Load validation exceptions ----------
try:
    ve = spark.table("validation_events").filter(
        (F.col("period") == period) & (F.col("run_id") == run_id) &
        (F.col("status") == "fail")
    )
    
    if ve.rdd.isEmpty():
        print(f"[nb_exception_triage] No validation failures for period={period}, run_id={run_id}")
        mssparkutils.notebook.exit("OK")
    
    exception_count = ve.count()
    print(f"[nb_exception_triage] Triaging {exception_count} exceptions...")

except Exception as e:
    print(f"[ERROR] Failed to load validation_events: {type(e).__name__}: {e}")
    sys.exit(1)

# ---------- AI triage ----------
triage_labels = []
success_count = 0
error_count = 0

try:
    for row in ve.collect():
        try:
            # Create event ID using hashlib (deterministic ID based on exception properties)
            event_id = hashlib.md5(
                f"{row['dfm_id']}{row['policy_id']}{row['rule_id']}{row['event_ts']}".encode()
            ).hexdigest()
            
            dfm_id = str(row["dfm_id"]) if row["dfm_id"] else "UNKNOWN"
            rule_id = str(row["rule_id"]) if row["rule_id"] else "UNKNOWN"
            details = str(row["details_json"]) if row["details_json"] else "{}"
            
            prompt = f"""
Triage this validation exception:
- DFM: {dfm_id}
- Rule: {rule_id}
- Status: fail
- Details: {details}

Classify as one of:
  - expected_design: This is normal/expected behavior
  - data_quality_issue: Source data problem
  - config_error: Configuration needs adjustment
  - requires_manual_review: Needs human decision
  - infrastructure_issue: System problem

Suggest priority (1=high, 5=low) and action.

Return JSON: {{
  "classification": "...",
  "root_cause": "...",
  "priority": 2,
  "action": "...",
  "escalate": false,
  "confidence": 0.8
}}
"""
            
            try:
                result_text = ai_client.triage_exception(prompt)
                
                if not result_text or not isinstance(result_text, str):
                    raise ValueError("AI returned non-string response")
                
                result = json.loads(result_text)
                
                triage_labels.append({
                    "period": period,
                    "run_id": run_id,
                    "dfm_id": dfm_id,
                    "validation_event_id": event_id,
                    "rule_id": rule_id,
                    "classification": str(result.get("classification", "requires_manual_review")),
                    "root_cause": str(result.get("root_cause", "")),
                    "suggested_action": str(result.get("action", "investigate")),
                    "priority": int(result.get("priority", 3)),
                    "ai_confidence": float(result.get("confidence", 0.0)),
                    "requires_escalation": bool(result.get("escalate", False)),
                    "triaged_at": None
                })
                success_count += 1
            
            except json.JSONDecodeError as e:
                error_count += 1
                print(f"[WARNING] AI response not valid JSON for {rule_id}: {e}")
                # Fallback: mark as requiring manual review
                triage_labels.append({
                    "period": period,
                    "run_id": run_id,
                    "dfm_id": dfm_id,
                    "validation_event_id": event_id,
                    "rule_id": rule_id,
                    "classification": "requires_manual_review",
                    "root_cause": f"AI triage failed: JSON parse error",
                    "suggested_action": "manual_investigation",
                    "priority": 3,
                    "ai_confidence": 0.0,
                    "requires_escalation": True,
                    "triaged_at": None
                })
            except Exception as e:
                error_count += 1
                print(f"[WARNING] Error processing {rule_id}: {type(e).__name__}: {e}")
                # Fallback: mark as requiring manual review
                triage_labels.append({
                    "period": period,
                    "run_id": run_id,
                    "dfm_id": dfm_id,
                    "validation_event_id": event_id,
                    "rule_id": rule_id,
                    "classification": "requires_manual_review",
                    "root_cause": f"AI triage failed: {type(e).__name__}",
                    "suggested_action": "manual_investigation",
                    "priority": 3,
                    "ai_confidence": 0.0,
                    "requires_escalation": True,
                    "triaged_at": None
                })
        
        except Exception as e:
            error_count += 1
            print(f"[WARNING] Failed to process exception row: {type(e).__name__}: {e}")

    # Write results
    if triage_labels:
        try:
            df_labels = spark.createDataFrame(triage_labels)
            df_labels.write.mode("append").saveAsTable("ai_triage_labels")
            print(f"[nb_exception_triage] COMPLETE wrote {len(triage_labels)} triage labels (success={success_count}, errors={error_count})")
        except Exception as e:
            print(f"[ERROR] Failed to write triage labels: {type(e).__name__}: {e}")
            sys.exit(1)
    else:
        print(f"[WARNING] No valid triage labels generated")
    
    mssparkutils.notebook.exit("OK")

except Exception as e:
    print(f"[ERROR] Exception triage failed: {type(e).__name__}: {e}")
    import traceback
    traceback.print_exc()
    sys.exit(1)